In [7]:
from __future__ import annotations

import operator
from typing import TypedDict, List, Annotated, Literal

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

In [8]:
class Task(BaseModel):
    id : int
    title : str
    goal : str = Field(
        ...,
        description = "One sentence regarding what the user shoukd underdstand after reading this section",
    )
    bullets : List[str] = Field(
        ...,
        min_length = 3 ,
        max_length  = 5 ,
        description = "List of 3-5 concrete , non-overlapping subpoints to cover this section",
    )
    target_words : int = Field(
        ... ,
        description="Target word count for this section(120-400).",
    )
    section_type : Literal[
        "common_mistakes",
        "introduction",
        "conclusion", "core" , "checklist" , "examples"
    ] = Field(
        ...,
        description = "Use 'common_mistakes' exactly once this section."
    )

In [9]:
class Plan(BaseModel):
    blog_title: str
    audience : str = Field(..., description="Target audience for the blog")
    tone : str = Field(... , description="Tone of the blog('practical' , 'crisp')")
    tasks : List[Task]

In [10]:
class State(TypedDict):
    topic  : str
    plan : Plan
    sections : Annotated[list[str], operator.add]
    final : str    


In [14]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv() # Loads variables from .env

llm = ChatGroq(model="qwen/qwen-2.5-32b")

In [15]:
def orchestrator(state  : State):
    plan = llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(
                content=(
                    "You are a senior technical writer and developer advocate. Your job is to produce a " "highly actionable outline for a technical blog post.\n\n" "Hard requirements:\n" "- Create 5-7 sections (tasks) that fit a technical blog.\n" "- Each section must include:\n" " 1) goal (1 sentence: what the reader can do/understand after the section)\n" " 2) 3-5 bullets that are concrete, specific, and non-overlapping\n" " 3) target word count (120-450)\n" "- Include EXACTLY ONE section with section_type='common_mistakes'.\n\n" "Make it technical (not generic):\n" "- Assume the reader is a developer; use correct terminology.\n" "- Prefer design/engineering structure: problem -> intuition -> approach -> implementation -> " "trade-offs -> testing/observability -> conclusion.\n" "- Bullets must be actionable and testable (e.g., 'Show a minimal code snippet for X', " "'Explain why Y fails under Z condition', 'Add a checklist for production readiness').\n" "- Explicitly include at least ONE of the following somewhere in the plan (as bullets):\n" " * a minimal working example (MWE) or code sketch\n" " * edge cases / failure modes\n" " * performance/cost considerations\n" " * security/privacy considerations (if relevant)\n" " * debugging tips / observability (logs, metrics, traces)\n" "- Avoid vague bullets like 'Explain X' or 'Discuss Y'. Every bullet should state what " "to build/compare/measure/verify.\n\n" "Ordering guidance:\n" "- Start with a crisp intro and problem framing.\n" "- Build core concepts before advanced details.\n" "- Include one section for common mistakes and how to avoid them.\n" "- End with a practical summary/checklist and next steps.\n\n" "Output must strictly match the Plan schema."
                )
            ),
            HumanMessage(content=f"Topic: {state['topic']}"),
        ]
    )
    return {"plan" : plan}

In [16]:
def fanout(state : State):
    return [Send("worker", {"task": task, "topic": state["topic"], "plan": state["plan"]})
            for task in state["plan"].tasks]

In [ ]:
def worker(payload: dict) -> dict:

    # payload contains what we sent
    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]

    blog_title = plan.blog_title

    section_md = llm.invoke(
        [
            SystemMessage(content="""You are a senior technical writer and developer advocate. Write ONE section of a technical blog post.

            Hard constraints:
        - Follow the provided Goal and cover ALL Bullets in order (do not skip or merge bullets).
- Stay close to the Target words (±15%).
- Output ONLY the section content in Markdown (no blog title H1, no extra commentary).

Technical quality bar:
- Be precise and implementation-oriented (developers should be able to apply it).
- Prefer concrete details over abstractions: APIs, data structures, protocols, and exact terms.
- When relevant, include at least one of:
  * a small code snippet (minimal, correct, and idiomatic)
  * a tiny example input/output
  * a checklist of steps
  * a diagram described in text (e.g., 'Flow: A -> B -> C')
- Explain trade-offs briefly (performance, cost, complexity, reliability).
- Call out edge cases / failure modes and what to do about them.
- If you mention a best practice, add the 'why' in one sentence.

Markdown style:
- Start with a '## <Section Title>' heading.
- Use short paragraphs, bullet lists where helpful, and code fences for code.
- Avoid fluff. Avoid marketing language.
- If you include code, keep it focused on the bullet being addressed.
""")
            HumanMessage(
                content=(
                    f"Blog: {blog_title}\n"
                    f"Topic: {topic}\n\n"
                    f"Section Title: {task.title}\n"
                    f"Goal: {task.goal}\n"
                    f"Bullets to cover:\n" + "\n".join([f"- {b}" for b in task.bullets]) + "\n\n"
                    f"Target word count: {task.target_words}\n"
                    f"Section Type: {task.section_type}\n\n"
                    "Return only the section content in Markdown."
                )
            ),
        ]
    ).content.strip()

    return {"sections": [section_md]}